In [2]:
import ollama
import httpx

llm_model = "gpt-oss:20b"

def wikipedia(q):
    response = httpx.get("https://en.wikipedia.org/w/api.php", params={
        "action": "query",
        "list": "search",
        "srsearch": q,
        "format": "json"
    })
    data = response.json()
    if "query" in data and "search" in data["query"] and len(data["query"]["search"]) > 0:
        return data["query"]["search"][0]["snippet"]
    else:
        return "No results found on Wikipedia."

class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        try:
            response = ollama.chat(
                model=llm_model,
                messages=self.messages,
                options={"temperature": 0},
                tools=[
                    {
                        "name": "wikipedia",
                        "description": "Search Wikipedia for information.",
                        "parameters": {
                            "type": "object",
                            "properties": {
                                "q": {
                                    "type": "string",
                                    "description": "The search query for Wikipedia."
                                }
                            },
                            "required": ["q"]
                        }
                    }
                ]
            )

            # Handle tool calls
            if "tool_calls" in response["message"]:
                for tool_call in response["message"]["tool_calls"]:
                    if tool_call["type"] == "function" and tool_call["function"]["name"] == "wikipedia":
                        q = tool_call["function"]["arguments"]["q"]
                        result = wikipedia(q)
                        self.messages.append({
                            "role": "tool",
                            "content": result,
                            "tool_call_id": tool_call["id"]
                        })
                        # Call execute again to get the final answer
                        return self.execute()
            return response["message"]["content"]

        except Exception as e:
            print(f"Error: {e}")
            return "Sorry, I encountered an error."

# Example usage
agent = Agent(system="You are a helpful assistant. You can use tools like Wikipedia to answer questions.")



In [3]:
response = agent(message="Hello, my name is Giovanni. How are you?")
print("Assistant:", response)

Assistant: Hello Giovanni! I’m doing great—thanks for asking. How can I help you today?


In [4]:
response = agent(message="What is the capital of Italy?")
print("Assistant:", response)

Assistant: The capital of Italy is **Rome**.


In [5]:
response = agent(message="What about Spain?")
print("Assistant:", response)

Assistant: The capital of Spain is **Madrid**.


In [6]:
response = agent(message="Thank you!")
print("Assistant:", response)

Assistant: You’re welcome! If you have any more questions, feel free to ask.
